# Pandas Benchmark

This notebook runs an representative end to end use case over data sourced from the [UK Land Registry House Price Data open data repository](https://www.gov.uk/government/statistical-data-sets/price-paid-data-downloads).

This data is made available for us under an [Open Government Licence](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/).

We will run two processes:

1. Load raw data, clean it up, add new features and finally write it as a mini dimensional model to lakehouse.
1. Query two of the tables in the dimensional model, join them and summarise the data.

In [1]:
import pandas as pd
import time
import logging
import os

In [2]:
logger = logging.getLogger(name="pandas_benchmark_notebook")
logger.setLevel(logging.INFO)

In [3]:
from datetime import datetime

source_path = "../../data/fabric/Files/land_registry_data" # ABFSS path to location where raw data (multiple CSV files) is stored
storage_options = {}

# Add timestamp to paths to avoid overwrite issues with Parquet
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
target_path_prices = f"../../data/fabric/Tables/pandas_benchmark/{run_timestamp}/prices.parquet"
target_path_locations = f"../../data/fabric/Tables/pandas_benchmark/{run_timestamp}/locations.parquet"
target_path_dates = f"../../data/fabric/Tables/pandas_benchmark/{run_timestamp}/dates.parquet"

In [4]:
start = time.perf_counter()

In [5]:
import glob
import os
import fsspec

logging.info(f"Reading price paid data from location {source_path}...")

# Define column names and types
column_names = [
    "transaction_unique_identifier",
    "price",
    "date_of_transfer",
    "postcode",
    "property_type",
    "old_new",
    "duration",
    "paon",
    "saon",
    "street",
    "locality",
    "town_city",
    "district",
    "county",
    "ppd_category_type",
    "record_status"
]

column_dtypes = {
    "transaction_unique_identifier": str,
    "price": float,
    "postcode": str,
    "property_type": str,
    "old_new": str,
    "duration": str,
    "paon": str,
    "saon": str,
    "street": str,
    "locality": str,
    "town_city": str,
    "district": str,
    "county": str,
    "ppd_category_type": str,
    "record_status": str
}

# Find all CSV files - handle both local and ABFSS paths
if source_path.startswith("abfss://"):
    # Use fsspec for cloud storage (ABFSS/OneLake)
    fs = fsspec.filesystem("abfss", **storage_options)
    all_files = fs.ls(source_path, detail=False)
    csv_files = [f"abfss://{f}" for f in all_files if f.endswith(".csv")]
else:
    # Use glob for local file system (filter out directories)
    csv_files = [f for f in glob.glob(os.path.join(source_path, "*.csv")) if os.path.isfile(f)]

logging.info(f"Found {len(csv_files)} CSV files: {csv_files}")

# Read and concatenate all CSV files
dfs = []
for csv_file in csv_files:
    df = pd.read_csv(
        csv_file,
        header=None,
        na_values=[""],
        names=column_names,
        dtype=column_dtypes,
        parse_dates=["date_of_transfer"],
        storage_options=storage_options if source_path.startswith("abfss://") else None,
    )
    dfs.append(df)

price_paid_data = pd.concat(dfs, ignore_index=True)
logging.info(f"Loaded {len(price_paid_data)} rows")

## Data Transformation

Now we have the DataFrame loaded, we can start to build up the transformations we want to apply:

In [6]:
# Convert the property_type column from single letter codes to full descriptions
property_type_mapping = {
    "D": "Detached",
    "S": "Semi-Detached",
    "T": "Terraced",
    "F": "Flat/Maisonette",
    "O": "Other"
}
price_paid_data["property_type"] = price_paid_data["property_type"].map(property_type_mapping).fillna(price_paid_data["property_type"])

In [7]:
# Do the same for old_new
old_new_mapping = {
    "Y": "New",
    "N": "Old"
}
price_paid_data["old_new"] = price_paid_data["old_new"].map(old_new_mapping).fillna(price_paid_data["old_new"])

In [8]:
# Use regex to extract the postcode area (the first one or two letters)
price_paid_data["postcode_area"] = price_paid_data["postcode"].str.extract(r"^([A-Z]{1,2})", expand=False)

In [9]:
# Convert date_of_transfer from datetime to date
price_paid_data["date_of_transfer"] = price_paid_data["date_of_transfer"].dt.date

### Create fact table

Select the core columns we want to use in the core fact table.

In [10]:
# Select relevant columns for downstream analysis
prices = price_paid_data[[
    "price",
    "date_of_transfer",
    "postcode_area",
    "town_city",
    "property_type",
    "old_new",
]].copy()

### Create date dimension

Use min and max dates to build date dimension table.

In [11]:
min_date = price_paid_data["date_of_transfer"].min()
max_date = price_paid_data["date_of_transfer"].max()
min_date, max_date

(datetime.date(2023, 1, 1), datetime.date(2025, 11, 28))

In [12]:
date_range = pd.date_range(start=min_date, end=max_date, freq="D")
dates = pd.DataFrame({"date": date_range})
dates["year"] = dates["date"].dt.year
dates["month"] = dates["date"].dt.month
dates["month_name"] = dates["date"].dt.strftime("%B")
dates["day"] = dates["date"].dt.day
dates["weekday"] = dates["date"].dt.weekday
dates["weekday_name"] = dates["date"].dt.strftime("%A")
dates["day_of_year"] = dates["date"].dt.dayofyear
dates["date"] = dates["date"].dt.date

### Create location dimension

Assumption is there is a hierarchy in descreasing order of granularity:

- County
- District
- Town or City

In [13]:
locations = price_paid_data[[
    "county",
    "district",
    "town_city",
]].drop_duplicates()

## Writing to Delta Tables

It is common practice to write out a Pandas DataFrame to a Delta table in the Tables area of your Lakehouse.

There are various write modes which are available:

Overwrite entire table:

```python
write_deltalake(path, df, mode="overwrite")
```

Append to existing table:

```python
write_deltalake(path, df, mode="append")
```

Merge (upsert) - use DeltaTable API:

```python
dt = DeltaTable(path)
(
    dt.merge(
        source=df,
        predicate="source.id = target.id",
        source_alias="source",
        target_alias="target"
    )
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute()
)
```

### Handling Timestamps

A common gotcha when writing Delta tables from Pandas is timezone handling. Fabric's SQL endpoint expects timestamps with timezone information.

We can address this by adding timezone information, for example:

```python
df["datetime_of_order"] = df["datetime_of_order"].dt.tz_localize("UTC")
```

### Write tables

In [14]:
logger.info(f"Writing prices data to Parquet: {target_path_prices}")
os.makedirs(os.path.dirname(target_path_prices), exist_ok=True)
prices.to_parquet(target_path_prices, index=False)

INFO:pandas_benchmark_notebook:Writing prices data to Parquet: ../../data/fabric/Tables/pandas_benchmark/20260120_182633/prices.parquet


In [15]:
logger.info(f"Writing locations data to Parquet: {target_path_locations}")
locations.to_parquet(target_path_locations, index=False)

INFO:pandas_benchmark_notebook:Writing locations data to Parquet: ../../data/fabric/Tables/pandas_benchmark/20260120_182633/locations.parquet


In [16]:
logger.info(f"Writing dates data to Parquet: {target_path_dates}")
dates.to_parquet(target_path_dates, index=False)

INFO:pandas_benchmark_notebook:Writing dates data to Parquet: ../../data/fabric/Tables/pandas_benchmark/20260120_182633/dates.parquet


## Reading from DeltaLake and generate summary

Let's illustrate this by generating some analytics in this notebook using the data we have just written to the lakehouse in Delta format.

In [17]:
# Load prices from Parquet and filter them to exclude "Other" property types
logger.info(f"Reading prices data back from Parquet: {target_path_prices}")
prices = pd.read_parquet(target_path_prices)
prices = prices[prices["property_type"] != "Other"]

INFO:pandas_benchmark_notebook:Reading prices data back from Parquet: ../../data/fabric/Tables/pandas_benchmark/20260120_182633/prices.parquet


In [18]:
# Load the date dimension, add a new month_tag column in the form YYYY_MM
logger.info(f"Reading dates data back from Parquet: {target_path_dates}")
dates = pd.read_parquet(target_path_dates)
dates["date"] = pd.to_datetime(dates["date"])
dates["month_tag"] = dates["date"].dt.strftime("%Y_%m")

INFO:pandas_benchmark_notebook:Reading dates data back from Parquet: ../../data/fabric/Tables/pandas_benchmark/20260120_182633/dates.parquet


In [19]:
# Now join the two tables to get month_tag into the prices table
prices["date_of_transfer"] = pd.to_datetime(prices["date_of_transfer"])
prices = prices.merge(
    dates[["date", "month_tag"]],
    left_on="date_of_transfer",
    right_on="date",
    how="left"
)

In [20]:
# Finally summarise the data up to monthly level by property type
monthly_summary = (
    prices
    .groupby(["month_tag", "property_type"])
    .agg(
        number_of_transactions=("price", "count"),
        median_price=("price", "median"),
        min_price=("price", "min"),
        max_price=("price", "max"),
    )
    .reset_index()
    .sort_values(["month_tag", "property_type"])
)

In [21]:
monthly_summary.head(5)

,month_tag,property_type,number_of_transactions,median_price,min_price,max_price
0,2023_01,Detached,13310,430000.0,33000.0,44750000.0
1,2023_01,Flat/Maisonette,12979,239000.0,14000.0,13500000.0
2,2023_01,Semi-Detached,16639,260000.0,950.0,28000000.0
3,2023_01,Terraced,17796,212500.0,12500.0,16150000.0
4,2023_02,Detached,13104,420000.0,39999.0,17500000.0


In [22]:
elapsed = time.perf_counter() - start
logger.info(f"Notebook completed in {elapsed:.2f} seconds.")

INFO:pandas_benchmark_notebook:Notebook completed in 9.49 seconds.


## Summary

Pandas is a widely-used data manipulation library that integrates well with Microsoft Fabric. While it loads data eagerly into memory (unlike Polars' lazy evaluation), it remains a practical choice for many data engineering workloads, especially when working with datasets that fit comfortably in memory.